# ناسورایا / تداش — آزمون کوتاه در Google Colab

این دفترچه محیط آزمایش را در Colab آماده می‌کند. کد آزموده‌شده داخل خود فایل است؛
به GitHub، فایل ZIP جدا، Google Drive mount، GPU یا نصب روی دستگاه شما نیاز ندارد.

**روش استفاده:** فایل را در [Google Colab](https://colab.research.google.com/) از
`File → Upload notebook` باز کنید. سلول‌ها را به ترتیب اجرا کنید. CPU کافی است.
سه سلول اول آزمایش بدون کلید هستند؛ سلول چهارم یک اجرای واقعی API انجام می‌دهد.

اجرای Colab رایگانِ محدود دارد؛ فراخوانی OpenAI API هزینهٔ جدا و کلید پروژهٔ دارای
مجوز می‌خواهد. این دفترچه سقف مالی دلاری اعمال نمی‌کند؛ حدود آن پنج کنش، چهل
فراخوانی ابزار و شصت ثانیه‌اند. حذف جلسه در قطع شبکه best effort است.

کلید را فقط در ورودی مخفی سلول اجرای زنده وارد کنید. مقدار کلید در کد، فایل، محیط
یا گزارش نوشته نمی‌شود. هنگام دانلود، فقط گزارش‌های اجرا بسته‌بندی می‌شوند.

این آزمایش تعاملی و موقت است. ماشین‌های Colab ممکن است هنگام بی‌کاری حذف شوند؛
پس از هر اجرا گزارش را دانلود کنید. هستهٔ تصمیم‌گیری، مدل خود و بدن تداش تغییر نمی‌کنند.

منابع: [راهنمای Colab](https://research.google.com/colaboratory/faq.html)،
[Agents API quickstart](https://developers.openai.com/api/docs/guides/agents-api/quickstart)،
[هزینه و مفهوم جلسه](https://developers.openai.com/api/docs/guides/agents-api/overview/).


## ۱. آماده‌سازی کد
این سلول بستهٔ تعبیه‌شده را با SHA-256 بررسی و در پوشه‌ای تازه باز می‌کند.
snapshot کد: `7cda92a41ae494bcfa00d5c9a0d48b2087722935`؛ شواهد موجود: ۴۲ آزمون موفق.


In [ ]:
import base64, hashlib, io, json, os, subprocess, sys, tempfile, uuid, zipfile
from pathlib import Path

assert sys.version_info >= (3, 10), 'Python 3.10 or later is required'
SOURCE_SNAPSHOT = '7cda92a41ae494bcfa00d5c9a0d48b2087722935'
SOURCE_SHA256 = '7022b339d7dde20a8e66ea190436acbf83995cfc34cda32cb2b2eb4ccb7fc3fd'
SOURCE_B64 = '...'
payload = base64.b64decode(SOURCE_B64)
assert hashlib.sha256(payload).hexdigest() == SOURCE_SHA256, 'Embedded code checksum mismatch'
ROOT = Path(tempfile.mkdtemp(prefix='nasuraya-colab-'))
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for info in archive.infolist():
        relative = Path(info.filename)
        assert not relative.is_absolute() and '..' not in relative.parts
        assert info.filename.startswith('examples/nasuraya_agent_loop/')
        assert ((info.external_attr >> 16) & 0o170000) != 0o120000
        archive.extract(info, ROOT)
DEPS = ROOT / 'runtime-deps'
BASE_ENV = dict(os.environ)
BASE_ENV.pop('OPENAI_API_KEY', None)
BASE_ENV['PYTHONPATH'] = str(DEPS) + os.pathsep + str(ROOT)
last_result_dir = None
print('کد بررسی و آماده شد؛ snapshot:', SOURCE_SNAPSHOT[:7])


## ۲. نصب وابستگی‌های اختیاری
وابستگی‌ها در پوشهٔ همین آزمایش نصب می‌شوند. بسته‌های محیط اصلی Colab تغییر نمی‌کنند.
نصب معمولاً مقداری زمان می‌برد؛ برای این آزمایش CPU کافی است.


In [ ]:
requirements = ROOT / 'examples/nasuraya_agent_loop/requirements-api.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
                '--target', str(DEPS), '-r', str(requirements)], check=True)
print('وابستگی‌ها نصب شدند.')


## ۳. آزمون کد و اجرای بدون کلید
اگر آزمون‌ها موفق باشند، سه کار ساختگی با پنج کنش انجام می‌شوند.
این سلول هزینهٔ OpenAI API ندارد و هیچ استنتاج مدلی اجرا نمی‌کند.


In [ ]:
tests = subprocess.run([sys.executable, '-m', 'unittest', 'discover',
                        '-s', 'examples/nasuraya_agent_loop/tests', '-v'],
                       cwd=ROOT, env=BASE_ENV, capture_output=True, text=True)
print(tests.stdout + tests.stderr)
assert tests.returncode == 0, 'Tests failed; inspect the output before live execution'
offline_dir = ROOT / ('offline-' + uuid.uuid4().hex[:8])
offline = subprocess.run([sys.executable, '-m', 'examples.nasuraya_agent_loop',
                          '--max-actions', '5', '--output-dir', str(offline_dir)],
                         cwd=ROOT, env=BASE_ENV, capture_output=True, text=True)
assert offline.returncode == 0, 'Offline execution failed'
last_result_dir = offline_dir
print(json.dumps(json.loads(offline.stdout), indent=2, ensure_ascii=False))
print('اجرای بدون کلید موفق شد.')


## ۴. اجرای زندهٔ کوتاه
مدل پیش‌فرض از quickstart رسمی است؛ دسترسی حساب شما به آن هنوز بررسی نشده.
در صورت داشتن مدل مجاز دیگری، نام آن را در فیلد MODEL وارد کنید.
سپس این سلول را اجرا و کلید پروژهٔ OpenAI را در ورودی مخفی وارد کنید.

مجوزهای موردنیاز طبق quickstart: `api.agents.read`، `api.agents.write` و
`api.responses.write`. دریافت یک کلید خالی هیچ درخواست API ایجاد نمی‌کند.
نتیجهٔ `completed` فقط وقتی ثبت می‌شود که نوبت اصلی تمام شده و هدف در وضعیت
میزبان واقعاً برآورده شده باشد. هر بار اجرای این سلول یک جلسهٔ تازه ایجاد می‌کند.


In [ ]:
#@title اجرای محدود با Agents API
MODEL = "gpt-6-astra" #@param {type:"string"}
LIVE_RUNNER = "import json, sys\nfrom pathlib import Path\nfrom openai import OpenAI\nfrom examples.nasuraya_agent_loop.agents_api import run_api\nfrom examples.nasuraya_agent_loop.local import Journal, State, ToolHost, write_report\n\ndata = json.load(sys.stdin)\ndestination = Path(data['output_dir'])\ndestination.mkdir(parents=True, exist_ok=False)\njournal = Journal(destination / 'events.jsonl')\nhost = ToolHost(State(), journal, max_actions=5)\ntry:\n    with OpenAI(api_key=data.pop('api_key'), base_url='https://api.openai.com/v1',\n                timeout=10.0, max_retries=0) as client:\n        result = run_api(host, client, model=data['model'], max_calls=40,\n                         timeout_seconds=60, live_service=True)\nexcept Exception as exc:\n    print(json.dumps({'status':'client_initialization_failed','exception_type':type(exc).__name__}))\n    raise SystemExit(2)\n(destination / 'summary.json').write_text(json.dumps(result,indent=2)+'\\n')\nwrite_report(destination / 'report.md', result, journal)\nprint(json.dumps(result))\nraise SystemExit(0 if result['status']=='completed' else 1)\n"
assert MODEL.strip(), 'Set a model available to your API project'
try:
    from IPython import get_ipython
    assert get_ipython() is not None, 'Run this cell in an interactive notebook'
    from getpass import getpass
    _secret = getpass('کلید OpenAI API — ورودی مخفی: ')
    if not _secret.strip():
        print('کلیدی وارد نشد؛ درخواست API ارسال نشد.')
    else:
        live_dir = ROOT / ('live-' + uuid.uuid4().hex[:8])
        live = subprocess.run([sys.executable, '-c', LIVE_RUNNER],
                              input=json.dumps({'api_key':_secret, 'model':MODEL.strip(),
                                                'output_dir':str(live_dir)}),
                              cwd=ROOT, env=BASE_ENV, capture_output=True, text=True)
        # Print parsed summaries only; raw stderr can contain request details.
        try:
            live_result = json.loads(live.stdout)
            print(json.dumps(live_result, indent=2, ensure_ascii=False))
        except json.JSONDecodeError:
            print('اجرای فرایند کامل نشد؛ خروجی قابل‌خواندن دریافت نشد.')
        if (live_dir / 'summary.json').exists():
            last_result_dir = live_dir
        if live.returncode == 0:
            print('اجرای زنده موفق شد؛ گزارش را با سلول بعد دانلود کنید.')
        else:
            print('موفقیت زنده تأیید نشد. وضعیت و گزارش موجود را بررسی کنید.')
finally:
    if '_secret' in globals():
        del _secret


## ۵. دریافت گزارش
فقط سه فایل گزارش آخرین اجرا دانلود می‌شوند: `events.jsonl`، `summary.json` و
`report.md`. اگر اجرای زنده انجام نشده باشد، گزارش اجرای بدون کلید دریافت می‌شود.
در قطع شبکه، رخداد `api_cleanup_failed` به معنی تأییدنشدن حذف جلسهٔ سمت سرویس است.


In [ ]:
assert last_result_dir is not None, 'Run the offline or live cell first'
report_zip = ROOT / (last_result_dir.name + '-reports.zip')
with zipfile.ZipFile(report_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in ('events.jsonl', 'summary.json', 'report.md'):
        path = last_result_dir / name
        if path.exists():
            archive.write(path, arcname=name)
print('گزارش آماده است:', report_zip.name)
try:
    from google.colab import files
    files.download(str(report_zip))
except ImportError:
    print('این محیط Colab نیست؛ فایل گزارش در مسیر چاپ‌شده قرار دارد:', report_zip)
